In [63]:
import sklearn
import numpy as np
import pandas as pd

### Data exploration

- Check out data shape, size, structure, missing values, ...
- ACP to determine most important variables
- Visualisation to observe trends



Load a second dataset with reviews, match by appid (should be universal for steam)
Price tag: free, cheap, normal price, expensive, very expensive. + discount tag 
correlation between: price, reviews. genre and reviews, company/publisher and price + reviews 

recs based on: reviews, genres, general sentiment from the reviews (type of game, style of gameplay, Vibe) 


EDA:
best sellign games/genres/publishers. Signs of client satisfaction. 

In [64]:
games=pd.read_json("data/games.json").T.reset_index()
games.rename(columns={"index":"app_id"},inplace=True)


In [65]:
games.info
games.columns.tolist()
print(games['genres'].head(3))
print(games['tags'].head(3))
print(games['movies'].head(5))
games.isnull().sum()


0             []
1    [Adventure]
2       [Casual]
Name: genres, dtype: object
0                                                   []
1    {'Adventure': 27, 'Visual Novel': 19, 'Anime':...
2    {'Casual': 83, 'Card Game': 52, 'Solitaire': 4...
Name: tags, dtype: object
0    []
1    []
2    []
3    []
4    []
Name: movies, dtype: object


app_id                      0
name                        0
release_date                0
required_age                0
price                       0
dlc_count                   0
detailed_description        0
about_the_game              0
short_description           0
reviews                     0
header_image                0
website                     0
support_url                 0
support_email               0
windows                     0
mac                         0
linux                       0
metacritic_score            0
metacritic_url              0
achievements                0
recommendations             0
notes                       0
supported_languages         0
full_audio_languages        0
packages                    0
developers                  0
publishers                  0
categories                  0
genres                      0
screenshots                 0
movies                      0
user_score                  0
score_rank                  0
positive  

In [70]:
mask = (
    (games['short_description'] == '') &
    (games['genres'].apply(lambda x: x == [])) &
    (games['tags'].apply(lambda x: x == [] or x == {}))
)
mask.sum()

games_clean = games[~mask].reset_index(drop=True)

games_clean=games_clean.drop(columns=["support_url","support_email","metacritic_score","metacritic_url","achievements","recommendations","notes","movies","average_playtime_2weeks","median_playtime_2weeks","peak_ccu"])

In [71]:
games_clean.head(5)

,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,screenshots,user_score,score_rank,positive,negative,estimated_owners,average_playtime_forever,median_playtime_forever,discount,tags
0,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,[https://shared.akamai.steamstatic.com/store_i...,0,,252,3,0 - 20000,8,8,65,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':..."
1,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,[https://shared.akamai.steamstatic.com/store_i...,0,,21,3,0 - 20000,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4..."
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0 - 20000,0,0,0,[]
3,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0 - 20000,0,0,0,[]
4,1654170,Agony VR,"Apr 5, 2023",0,13.99,0,ADD TO WISHLIST About the Game A JOURNEY THROU...,A JOURNEY THROUGH HELL! Explore the most terri...,Agony VR is a first-person survival horror gam...,,...,[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0 - 20000,0,0,0,[]


TODO: 
- Filtrer les jeux complètement vides
- Regrouper windows, mac, linux → colonne os
- Créer la colonne price_range (free / cheap / moderate / expensive)
- Convertir estimated_owners de string "0 - 20000" vers quelque chose d'utilisable (moyenne ou borne haute)
- Créer une colonne total_reviews = positive + negative
- Créer une colonne positive_ratio = positive / total_reviews (attention division par zéro)

In [72]:
def get_os(row):
    os_list=[]
    if row["windows"]:
        os_list.append("windows")
    if row["linux"]:
        os_list.append("linux")
    if row["mac"]:
        os_list.append("mac")
    return os_list

games_clean["os"]=games_clean.apply(get_os, axis=1)
games_clean.drop(columns=["windows","mac","linux","discount","estimated_owners"])

,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,genres,screenshots,user_score,score_rank,positive,negative,average_playtime_forever,median_playtime_forever,tags,os
0,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,[Adventure],[https://shared.akamai.steamstatic.com/store_i...,0,,252,3,8,8,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':...",[windows]
1,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,[Casual],[https://shared.akamai.steamstatic.com/store_i...,0,,21,3,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4...","[windows, mac]"
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,"[Casual, Indie, Simulation]",[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0,0,[],[windows]
3,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,"[Action, Early Access]",[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0,0,[],[windows]
4,1654170,Agony VR,"Apr 5, 2023",0,13.99,0,ADD TO WISHLIST About the Game A JOURNEY THROU...,A JOURNEY THROUGH HELL! Explore the most terri...,Agony VR is a first-person survival horror gam...,,...,"[Action, Adventure]",[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0,0,[],[windows]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114336,4152910,完美传奇,"Jan 4, 2026",0,0.0,0,《完美传奇》游戏介绍 🔥【完美传奇——专属大陆的奇幻史诗！】🔥 🌍耗时两年匠心打造，诚意之作...,《完美传奇》游戏介绍 🔥【完美传奇——专属大陆的奇幻史诗！】🔥 🌍耗时两年匠心打造，诚意之作...,欢迎来到《完美传奇》！ 🌍专属大陆×自由探索，六道轮回剧情×百变BUFF搭配，一刀爆神装、机...,,...,"[Action, Adventure, Massively Multiplayer, RPG...",[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0,0,[],[windows]
114337,4042800,Poker Fate - ACG Texas Hold'em,"Jan 3, 2026",0,0.0,0,Poker Fate – Choose your poker partner and ent...,Poker Fate – Choose your poker partner and ent...,Poker Fate is an anime-style Texas Hold'em gam...,,...,"[Casual, Strategy, Free To Play]",[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0,0,[],[windows]
114338,3522550,Adira Nusantara,"Jan 3, 2026",0,7.99,0,(Gif character) Adira Nusantara is a side-scro...,(Gif character) Adira Nusantara is a side-scro...,"Master authentic Silat combat as Adira, a fier...",,...,"[Action, Adventure, Casual, Early Access]",[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0,0,[],[windows]
114339,3680350,A Lenda de Niterói,"Jan 4, 2026",0,2.09,0,"Step into the role of Arariboia, a legendary I...","Step into the role of Arariboia, a legendary I...",Embark on Arariboia’s journey during the 16th-...,,...,"[Action, Adventure, Casual, Indie]",[https://shared.akamai.steamstatic.com/store_i...,0,,0,0,0,0,[],[windows]


In [73]:
games_clean["total_reviews"]=games_clean["positive"]+games_clean["negative"]
games_clean["positive"] / games_clean["total_reviews"].replace(0, np.nan)
games_clean.head(5)

,app_id,name,release_date,required_age,price,dlc_count,detailed_description,about_the_game,short_description,reviews,...,score_rank,positive,negative,estimated_owners,average_playtime_forever,median_playtime_forever,discount,tags,os,total_reviews
0,496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0,5.24,0,"Springtime, April: when the cherry trees come ...","Springtime, April: when the cherry trees come ...","Spring has come, and our protagonist, Yukinari...",,...,,252,3,0 - 20000,8,8,65,"{'Adventure': 27, 'Visual Novel': 19, 'Anime':...",[windows],255
1,1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0,4.99,0,"Immerse yourself in the most beloved, mystical...","Immerse yourself in the most beloved, mystical...",Discover an entrancing and spectacular world!,,...,,21,3,0 - 20000,0,0,0,"{'Casual': 83, 'Card Game': 52, 'Solitaire': 4...","[windows, mac]",24
2,3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0,8.99,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...","synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",Yuha! I'll start the broadcast! Hakko's extrem...,,...,,0,0,0 - 20000,0,0,0,[],[windows],0
3,3631080,Maze Quest VR,"Apr 24, 2025",0,4.99,0,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,Its not just a Maze; its a Quest! Enter the ca...,,...,,0,0,0 - 20000,0,0,0,[],[windows],0
4,1654170,Agony VR,"Apr 5, 2023",0,13.99,0,ADD TO WISHLIST About the Game A JOURNEY THROU...,A JOURNEY THROUGH HELL! Explore the most terri...,Agony VR is a first-person survival horror gam...,,...,,0,0,0 - 20000,0,0,0,[],[windows],0
